In [2]:
import cv2
import numpy as np
import shutil
from pathlib import Path

 #CÂN BẰNG SÁNG (CLAHE)
def apply_clahe(img):
    # Chuyển sang hệ LAB để chỉ can thiệp vào độ sáng (kênh L), không làm hỏng màu biển báo
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    cl = clahe.apply(l)
    
    limg = cv2.merge((cl, a, b))
    return cv2.cvtColor(limg, cv2.COLOR_LAB2BGR)

# PHẦN 5: TĂNG CƯỜNG DỮ LIỆU (AUGMENTATION)
def add_brightness(img, value=30):
    # Tăng sáng mô phỏng trời nắng gắt
    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    h, s, v = cv2.split(hsv)
    
    v = cv2.add(v, value)
    v[v > 255] = 255
    v[v < 0] = 0
    
    final_hsv = cv2.merge((h, s, v))
    return cv2.cvtColor(final_hsv, cv2.COLOR_HSV2BGR)

def add_blur(img):
    # Làm mờ mô phỏng camera rung lắc hoặc xe chạy nhanh
    return cv2.GaussianBlur(img, (5, 5), 0)

def add_noise(img):
    # Thêm nhiễu hạt mô phỏng camera quay ban đêm chất lượng kém
    row, col, ch = img.shape
    gauss = np.random.normal(0, 15, (row, col, ch)).astype(np.float32)
    img_noisy = img.astype(np.float32) + gauss
    return np.clip(img_noisy, 0, 255).astype(np.uint8)


# THỰC THI TRÊN THƯ MỤC DỮ LIỆU


# 1. Cấu hình đường dẫn
PROJECT_ROOT = Path.cwd().parent
INPUT_IMAGES = PROJECT_ROOT / "dataset" / "dataset" / "train" / "images"
INPUT_LABELS = PROJECT_ROOT / "dataset" / "dataset" / "train" / "labels"

# Thư mục đích: chứa dữ liệu đã hoàn thiện sẵn sàng cho YOLO
OUTPUT_DIR = PROJECT_ROOT / "dataset" / "yolo_ready_train"
OUTPUT_IMAGES = OUTPUT_DIR / "images"
OUTPUT_LABELS = OUTPUT_DIR / "labels"

OUTPUT_IMAGES.mkdir(parents=True, exist_ok=True)
OUTPUT_LABELS.mkdir(parents=True, exist_ok=True)

TARGET_SIZE = (416, 416)

print("Bắt đầu dây chuyền tiền xử lý và tăng cường dữ liệu...")
count = 0

# Quét qua ảnh (bạn có thể xóa [0:100] để chạy toàn bộ dataset)
for img_path in list(INPUT_IMAGES.glob("*.*"))[0:100]:
    if img_path.suffix.lower() in ['.jpg', '.png', '.jpeg']:
        
        # Đọc ảnh gốc
        img_raw = cv2.imread(str(img_path))
        label_name = img_path.stem + ".txt"
        input_label_path = INPUT_LABELS / label_name
        
        if img_raw is not None and input_label_path.exists():
            
            # --- BƯỚC 1: RESIZE ẢNH GỐC ---
            img = cv2.resize(img_raw, TARGET_SIZE)
            
            # --- BƯỚC 2: TẠO CÁC PHIÊN BẢN (Gốc đã resize, CLAHE, Sáng, Mờ, Nhiễu) ---
            versions = {
                "ori": img,
                "clahe": apply_clahe(img),
                "bright": add_brightness(img),
                "blur": add_blur(img),
                "noise": add_noise(img)
            }
            
            # --- BƯỚC 3: LƯU ẢNH VÀ NHÂN BẢN FILE NHÃN ---
            for suffix, modified_img in versions.items():
                # Lưu ảnh
                new_img_name = f"{img_path.stem}_{suffix}.jpg"
                cv2.imwrite(str(OUTPUT_IMAGES / new_img_name), modified_img)
                
                # Copy file nhãn .txt (vì tọa độ YOLO là tỷ lệ % nên không bị hỏng khi resize)
                new_label_name = f"{img_path.stem}_{suffix}.txt"
                shutil.copy(input_label_path, OUTPUT_LABELS / new_label_name)
            
            count += 1
print(f"Tổng số ảnh thu được sẵn sàng cho YOLO: {count * 5} ảnh.")

Bắt đầu dây chuyền tiền xử lý và tăng cường dữ liệu...
Tổng số ảnh thu được sẵn sàng cho YOLO: 500 ảnh.
